# XGBoost OBS 推理服务 — 端到端验证（健康检查 → 基线推理 → 热切换闭环）

本 notebook 验证完整的 **训练 → 部署 → 推理 → 热切换** 流程。

## 章节

| 章节 | 内容 |
|---|---|
| §1 | 配置（⚠️ 必改） |
| §2 | 训练两套模型（旧=基线，新=验证热切换） |
| §3 | 健康检查 `GET /health` |
| §4 | 基线推理 `POST /` |
| **§5** | **热切换验证（核心）** |
| §6 | 排障 |

## 前置条件

- 推理服务已启动：
  - 本地 docker：`./build_and_run.ps1 test-obs -Ak <AK> -Sk <SK>`（OBS API 模式）
  - 云上 ModelArts：已完成部署，环境变量 `OBS_BUCKET`/`AccessKeyID`/`SecretAccessKey` 配齐
- 凭证：ModelArts API Key（云上推理用）+ OBS AK/SK（替换模型用）
  - 推荐用环境变量传入（`MODELARTS_API_KEY`、`OBS_AK`、`OBS_SK`），没有就运行时交互输入

## 0. 依赖

In [ ]:
import subprocess, sys
for pkg in ("xgboost", "scikit-learn", "pandas", "esdk-obs-python", "requests"):
    try:
        __import__(pkg.replace("-", "_") if pkg != "esdk-obs-python" else "obs")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
print("依赖就绪")

## 1. 配置（⚠️ 必改）

把 `<你的服务地址>` / `<服务ID>` / `<你的桶名>` 换成实际值，或用环境变量
`MODELARTS_OBS_INFER_URL` / `OBS_BUCKET` 覆盖。

In [ ]:
import getpass
import os
import shutil
from pathlib import Path

import json
import time
import requests
import urllib3

# === 服务地址 ===
# 云上 ModelArts: https://<你的服务地址>/v2/infer/<服务ID>/
# 本地 docker:    http://127.0.0.1:18081/
INFER_URL = os.environ.get(
    "MODELARTS_OBS_INFER_URL",
    "https://<你的服务地址>/v2/infer/<服务ID>/").strip().rstrip("/") + "/"

# === OBS 配置（云上热切换用）===
OBS_BUCKET   = os.environ.get("OBS_BUCKET", "<你的桶名>")
OBS_KEY      = "models/xgboost_breast_cancer.json"
OBS_ENDPOINT = "https://obs.cn-north-4.myhuaweicloud.com"

# === 本地路径（相对本 notebook 所在目录）===
SCRIPT_DIR       = Path.cwd()
REQUEST_PATH     = SCRIPT_DIR / "sample_request.json"
OLD_MODEL        = SCRIPT_DIR / "model_out" / "old" / "xgboost_breast_cancer.json"
NEW_MODEL        = SCRIPT_DIR / "model_out" / "new" / "xgboost_breast_cancer.json"
LOCAL_MOUNT      = SCRIPT_DIR / "model_mount"
LOCAL_MOUNT_FILE = LOCAL_MOUNT / "xgboost_breast_cancer.json"

TIMEOUT    = 30.0
VERIFY_TLS = False   # 自签名证书的私有化环境用 False；公网服务可改 True
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# === 校验 ===
assert "<" not in INFER_URL and "<" not in OBS_BUCKET, (
    "请先把 <你的服务地址>/<服务ID>/<你的桶名> 换成实际值"
    "（或设环境变量 MODELARTS_OBS_INFER_URL / OBS_BUCKET）")
assert REQUEST_PATH.exists(), f"找不到: {REQUEST_PATH}"

is_local = "127.0.0.1" in INFER_URL or "localhost" in INFER_URL

if is_local:
    AUTH_HEADERS = {"Content-Type": "application/json"}
else:
    API_KEY = os.environ.get("MODELARTS_API_KEY", "").strip()
    if not API_KEY:
        API_KEY = getpass.getpass("ModelArts API Key: ").strip()
    AUTH_HEADERS = {"Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"}

body = json.loads(REQUEST_PATH.read_text(encoding="utf-8"))

print(f"URL    = {INFER_URL}")
print(f"MODE   = {'本地 docker' if is_local else '云上 ModelArts'}")
print(f"OBS    = obs://{OBS_BUCKET}/{OBS_KEY}")

## 2. 训练两套模型

两套模型超参不同、预测值不同，这样替换后能明显看出热切换生效。
已跑过且 `model_out/` 里已有产物的话，本节可直接跳过。

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print(f"数据集: {X.shape[0]} 样本, {X.shape[1]} 特征")

def train_and_save(params, random_state, output_path, label):
    """训练、评估并保存模型。"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y)
    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", n_jobs=-1, random_state=random_state, **params)
    model.fit(X_train, y_train, verbose=False)

    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    print(f"  {label}: Accuracy={acc:.4f} AUC={auc:.4f} "
          f"-> {output_path} ({output_path.stat().st_size:,} bytes)")

if not (OLD_MODEL.exists() and NEW_MODEL.exists()):
    print("训练旧模型（基线）...")
    train_and_save(
        dict(n_estimators=100, max_depth=3, learning_rate=0.1,
             subsample=0.8, colsample_bytree=0.8),
        random_state=42, output_path=OLD_MODEL, label="OLD")
    print("训练新模型...")
    train_and_save(
        dict(n_estimators=250, max_depth=6, learning_rate=0.01,
             subsample=0.6, colsample_bytree=0.5, min_child_weight=5,
             reg_alpha=0.5, reg_lambda=2.0, gamma=0.5),
        random_state=2024, output_path=NEW_MODEL, label="NEW")
else:
    print(f"已有模型，跳过训练: {OLD_MODEL.stat().st_size:,}B / {NEW_MODEL.stat().st_size:,}B")

## 3. 健康检查 `GET /health`

In [ ]:
resp = requests.get(INFER_URL.rstrip("/") + "/health",
                    headers=AUTH_HEADERS, timeout=TIMEOUT, verify=VERIFY_TLS)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

h = resp.json()
assert h["status"] == "ok"
assert h["feature_count"] == 30
print(f"\nsync_mode    = {h['sync_mode']}")
print(f"model_origin = {h['model_origin']}")
if h["sync_mode"] == "obs-api":
    assert h["model_source"].startswith("obs://"), (
        f"obs-api 模式但模型没从 OBS 同步: {h.get('obs', {}).get('error')}")
print("\nHEALTH OK ✅")

## 4. 基线推理 `POST /`

In [ ]:
resp = requests.post(INFER_URL, headers=AUTH_HEADERS, json=body,
                    timeout=TIMEOUT, verify=VERIFY_TLS)
resp.raise_for_status()
pred_baseline = float(resp.json()[0]["predictresult"])
print(f"predictresult = {pred_baseline:.16f}")

## 5. 热切换验证（核心）

依次替换模型，验证 `predictresult` 自动变化。

**替换策略**：先删除 → 再上传（确保 OBS 文件系统感知到变化）。

In [ ]:
obs_client = None

if not is_local:
    # 云上：用 OBS API 替换对象（先 deleteObject → 再 putFile，两步走）
    from obs import ObsClient
    ak = os.environ.get("OBS_AK", "") or getpass.getpass("OBS AK: ")
    sk = os.environ.get("OBS_SK", "") or getpass.getpass("OBS SK: ")
    obs_client = ObsClient(access_key_id=ak, secret_access_key=sk,
                           server=OBS_ENDPOINT, timeout=30)

def replace_model(src_path):
    """替换模型：先删除旧文件/对象，再写入新的。"""
    if is_local:
        # 本地：先删 → 再复制（需要容器以 -v model_mount:/opt/model 挂载）
        LOCAL_MOUNT.mkdir(parents=True, exist_ok=True)
        if LOCAL_MOUNT_FILE.exists():
            LOCAL_MOUNT_FILE.unlink()
            print("  [本地] 已删除旧文件")
        shutil.copy2(str(src_path), str(LOCAL_MOUNT_FILE))
        print(f"  [本地] 已写入: {src_path.name} ({src_path.stat().st_size:,} bytes)")
    else:
        resp = obs_client.deleteObject(OBS_BUCKET, OBS_KEY)
        print(f"  [OBS] delete status={resp.status}: obs://{OBS_BUCKET}/{OBS_KEY}")
        resp = obs_client.putFile(OBS_BUCKET, OBS_KEY, str(src_path))
        assert resp.status < 300, (
            f"putFile failed: status={resp.status}, "
            f"errorCode={getattr(resp, 'errorCode', '')}, "
            f"errorMessage={getattr(resp, 'errorMessage', '')}")
        print(f"  [OBS] 已上传: {src_path.name} ({src_path.stat().st_size:,} bytes)")

def infer_once(label=""):
    resp = requests.post(INFER_URL, headers=AUTH_HEADERS, json=body,
                         timeout=TIMEOUT, verify=VERIFY_TLS)
    resp.raise_for_status()
    p = float(resp.json()[0]["predictresult"])
    tag = f" [{label}]" if label else ""
    print(f"  predictresult{tag} = {p:.16f}")
    return p

print("初始化完成 ✅")

In [ ]:
# 步骤 1: 替换为旧模型
print("=" * 55)
print("替换为【旧模型】")
print("=" * 55)
replace_model(OLD_MODEL)
time.sleep(1 if is_local else 3)
pred_old = infer_once("旧模型")

In [ ]:
# 步骤 2: 替换为新模型
print("=" * 55)
print("替换为【新模型】")
print("=" * 55)
replace_model(NEW_MODEL)
time.sleep(1 if is_local else 3)
pred_new = infer_once("新模型")

In [ ]:
# 步骤 3: 结果对比
diff = abs(pred_new - pred_old)
print("=" * 55)
print(f"  旧模型: {pred_old:.16f}")
print(f"  新模型: {pred_new:.16f}")
print(f"  差异:   {diff:.10f}")
print("=" * 55)

if diff > 1e-6:
    print("\n  ✅ 热切换成功！未重启服务，predictresult 自动变化")
else:
    print("\n  ⚠️ 热切换未生效")
    raise AssertionError(f"diff={diff}")

if obs_client:
    obs_client.close()

## 6. 排障

| 现象 | 原因 | 解决 |
|---|---|---|
| 云上替换 OBS 对象后热切换不生效 | OBS 文件系统 mtime 不刷新 | 用先 delete → 再 putFile 两步走（本 notebook 已实现） |
| 本地 docker 热切换不生效 | 没以 `-v` 挂载 model_mount 目录 | 重建容器加 `-v model_mount:/opt/model` |
| `/health` 的 `model_source` 不以 `obs://` 开头 | 没进 OBS API 模式（`OBS_BUCKET`/AK/SK 不齐） | 看返回里的 `sync_mode_reason`，缺哪个补哪个后重启 |
| `[obs-probe] FAILED status=403` | AK/SK 无效或无桶权限 | 核对凭证与桶策略；服务会降级到内置兜底模型继续运行 |
| 推理 401/403 | API Key 无效 | 检查 `Authorization: Bearer <Token>` |